In [1]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from xgboost import XGBRegressor
import pandas as pd

In [2]:
taxiTreino_clima = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/taxiTreino_clima.csv')

In [ ]:
def preparar_dados(df):
    # Garante que a coluna de data é datetime
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    
    # Arredonda a hora
    df['data_hora'] = df['lpep_pickup_datetime'].dt.floor('H')
    
    # Agrupa por hora
    df_agrupado = df.groupby('data_hora').agg(
        demanda_corridas=('lpep_pickup_datetime', 'count'),
        temperatura=('temperatura [C°]', 'mean'),
        umidade=('umidade relativa [%]', 'mean'),
        precipitacao=('precipitação acumulada em 1 hora [mm]', 'mean'),
        vento_vel=('velocidade do vento [mph]', 'mean'),
        feriado_fds=('feriado/fim de semana', 'max')
    ).reset_index()
    
    # Criar variáveis de tempo
    df_agrupado['hora_do_dia'] = df_agrupado['data_hora'].dt.hour
    df_agrupado['dia_da_semana'] = df_agrupado['data_hora'].dt.dayofweek
    df_agrupado['mes'] = df_agrupado['data_hora'].dt.month
    
    # Remove qualquer linha que tenha ficado com dados nulos no clima
    df_agrupado = df_agrupado.dropna()
    
    return df_agrupado

In [4]:
treino_agg = preparar_dados(taxiTreino_clima)


features = [
    'temperatura',
    'umidade',
    'precipitacao',
    'vento_vel', 
    'feriado_fds',
    'hora_do_dia',
    'dia_da_semana',
    'mes'
]

X_train = treino_agg[features]
y_train = treino_agg['demanda_corridas']

In [ ]:
# Modelo Base
modelo_base = XGBRegressor( 
    random_state=15
)

# Espaço de Busca Bayesiano
param_bayes = {
    'n_estimators': Integer(200, 1000),
    'max_depth': Integer(4, 9),
    'min_child_weight': Integer(1, 15),
    'subsample': Real(0.1, 1.0),
    'num_parallel_tree': Integer(1, 15),
    'learning_rate': Real(0.001, 0.1, prior='log-uniform')
}

# Configuração da Otimização Bayesiana
busca_bayesiana = BayesSearchCV(
    estimator=modelo_base,
    search_spaces=param_bayes,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1,
    random_state=15,
    verbose=3
)

# Treinamento
busca_bayesiana.fit(X_train, y_train)

# Resultados
print("\n###### BUSCA BAYESIANA CONCLUÍDA ######")
print("A melhor combinação encontrada foi:")
print(busca_bayesiana.best_params_)

# Melhor modelo
melhor_modelo_bayes = busca_bayesiana.best_estimator_

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi